In [ ]:
#单元格1
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from trainTest.metrics.get_test_metrics import PlotConfusionMatrix
import torch.nn.functional as F
import warnings

warnings.filterwarnings("ignore")

# --- 关键：确保项目根目录在路径中 ---
path = os.path.join(os.getcwd(), '..', '..')
sys.path.append(path)

# --- 导入您的项目文件 ---
from utils.common_utils import printlog, set_seed, make_dir, calculate_and_save_metrics
from utils.common_params import *  #
from models.LD_Net import LD_Net

# --- 💡 关键修正：导入 CNNRNNs 类 ---
from models.CNNRNNs import CNNRNNs

from trainTest.early_stopping.early_stopping import EarlyStopping
from trainTest.optimizers.get_optimizer import Optimizer
from trainTest.losses.get_loss import get_classification_loss
from trainTest.lr_schedulers.get_lr_scheduler import LrScheduler
from trainTest.metrics.metrics_utils import cal_metrics_torch

# 设置随机种子
set_seed(seed=42)
print("所有依赖导入成功。")

In [ ]:
#单元格2
# === 1. 数据集选择 (修复 NameError) ===
DATASET_NAME = "SIAT"
MODEL_TYPE = 'SC_BiGRU'  # 我们用这个名字来代表 CNNRNNs(rnn_type='BiGRU')
DENOISE_METHOD = 'LD_Net_Online_Classification(LongTrain)'  # 文件夹名
n_channels_check = 9  # SIAT 9 通道

# === 2. 关键路径设置 ===

# --- 💡 关键修正 1: LD-Net (降噪器) 的路径 ---
LD_NET_MODEL_PATH = os.path.join(
    os.getcwd(),
    'checkpoints', 'LD_Net_SIAT', 'ld_net_best_model.pth'
)

# --- 💡 关键修正 2: *噪声输入* (分类器输入) 的路径 ---
base_data_path = os.path.join(path, "preProcessing")  # 'path' 是根目录
DATA_DIR = os.path.join(
    base_data_path, "SIAT_LLMD_trainData", "Denoising_TrainSet_XY"
)

# --- 3. 模型与训练设置 ---
BATCH_SIZE = 64
LEARNING_RATE = 5e-4
EPOCHS = 200
PATIENCE = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === 4. 检查参数 ===
print(f"--- 步骤四: LD-Net + {MODEL_TYPE} ---")
print(f"设备: {DEVICE}")
print(f"将加载 LD-Net 降噪器: {LD_NET_MODEL_PATH}")
print(f"将加载 *成对数据* (X, Y) 源: {DATA_DIR}")
if C != n_channels_check:
    print(f"⚠️ 警告: common_params.py 中的 C={C} 与 SIAT (C=9) 不匹配!")
else:
    print("✅ C 参数检查通过。")

# === 5. 结果保存路径 (💡 关键修正) ===
# 1. 创建包含模型名称的根目录 (例如: .../LD_Net_Online_Classification/SC_BiGRU/)
save_dir_root = os.path.join(path, 'results', DENOISE_METHOD, MODEL_TYPE)
# 2. 让模型目录和根目录是同一个地方
save_dir_models_temp = save_dir_root # <-- 指向同一个路径
make_dir(save_dir_root)

print(f"✅ 最终结果 (CSV/PNG) 将保存到: {save_dir_root}")
print(f"✅ 临时模型 (PTH) 将保存到: {save_dir_models_temp}")

In [ ]:
#单元格3
print("--- 正在加载 LD-Net 降噪模型 (步骤三的结果) ---")
try:
    model_denoise = torch.load(LD_NET_MODEL_PATH, weights_only=False)
    model_denoise.to(DEVICE)
    model_denoise.eval()
    print("✅ LD-Net 降噪模型加载成功并设为 eval() 模式。")
except Exception as e:
    print(f"❌ 加载 LD-Net 模型失败: {e}")
    print("请确保 步骤三 已成功运行，且 LD_NET_MODEL_PATH 路径正确。")
    raise e

In [ ]:
#单元格4
import glob


class LdNetClassificationDataset(Dataset):
    """
    用于分类器训练的数据集。
    从 步骤一 的 'Denoising_TrainSet_XY' 文件夹加载数据。
    """

    def __init__(self, data_dir, subject_id):
        self.data_dir = data_dir
        self.subject_id = subject_id

        # --- 匹配 'Denoising_TrainSet_XY' 的文件名 ---
        file_name_prefix = '_XY_TrainData.npz'
        search_path = os.path.join(self.data_dir, f"{self.subject_id}{file_name_prefix}")
        file_paths = glob.glob(search_path)

        if not file_paths:
            print(f"警告: 在 {self.data_dir} 中未找到 {self.subject_id}{file_name_prefix}。")
            self.samples = np.empty((0, C, window))
            self.labels_encoded = np.empty((0,))
            return

        try:
            data = np.load(file_paths[0])
            # 步骤一 保存的键是 'noisy_X' 和 'sub_motion_label_encoded'
            self.samples = data['noisy_X'].astype(np.float32)
            self.labels_encoded = data['sub_motion_label_encoded'].astype(np.int64)
            print(f"  已加载 {self.subject_id}: {self.samples.shape[0]} 个样本。")

        except Exception as e:
            print(f"加载 {file_paths[0]} 出错: {e}")
            self.samples = np.empty((0, C, window))
            self.labels_encoded = np.empty((0,))

    def __len__(self):
        return self.samples.shape[0]

    def __getitem__(self, idx):
        # 返回 噪声X 和 标签Y
        return self.samples[idx], self.labels_encoded[idx]


print("✅ 分类器数据集 (LdNetClassificationDataset) 定义完毕。")

In [ ]:
# 单元格5 (已集成 补丁 1, 2, 3, 4)
from sklearn.model_selection import train_test_split
from trainTest.metrics.get_test_metrics import PlotConfusionMatrix
import torch.optim as optim
import numpy as np
import torch.nn as nn
from trainTest.metrics.metrics_utils import (
    get_accuracy, get_precision, get_recall, get_f1, get_specificity
)
from sklearn.metrics import roc_auc_score
# --- (本地 cal_metrics_torch 定义，已修复 TypeError 和 AttributeError) ---
def cal_metrics_torch(y_true, y_pre, y_scores=None):
    """
    计算分类任务的所有指标 (本地定义)
    """
    if isinstance(y_true, torch.Tensor):
        y_true = y_true.cpu().numpy()
    if isinstance(y_pre, torch.Tensor):
        y_pre = y_pre.cpu().numpy()

    metrics = {
        'Accuracy': get_accuracy(y_true, y_pre),
        'Precision': get_precision(y_true, y_pre),
        'Recall': get_recall(y_true, y_pre),
        'F1': get_f1(y_true, y_pre),
        'Specificity': get_specificity(y_true, y_pre),
    }

    if y_scores is not None:
        if isinstance(y_scores, torch.Tensor):
            y_scores = y_scores.cpu().numpy()

        if not isinstance(y_scores, np.ndarray):
            try:
                y_scores = np.stack(y_scores, axis=0)
            except (ValueError, TypeError):
                y_scores = np.array([])

        if len(y_scores.shape) == 2 and y_scores.shape[1] > 1:
            try:
                auc = roc_auc_score(y_true, y_scores, multi_class='ovr', average='macro')
                metrics['AUC'] = round(auc * 100.0, 3)
            except ValueError as e:
                metrics['AUC'] = 0.0
        else:
            metrics['AUC'] = 0.0

    return metrics


# --- 修正结束 ---


def train_one_epoch(model_classifier, model_denoiser, train_loader, criterion, optimizer, device):
    model_classifier.train()
    train_loss = 0.
    total_num = 0
    y_true_all = []
    y_pred_all = []

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # --- 实时降噪 ---
        with torch.no_grad():
            out = model_denoiser(inputs)
            inputs_denoised = out[0] if isinstance(out, tuple) else out
        # --- 结束 ---

        # --- 💡 补丁 4 (可选)：轻量数据增强 (Time Shift) ---
        if model_classifier.training:
            shift = torch.randint(-5, 6, (1,)).item()
            inputs_denoised = torch.roll(inputs_denoised, shifts=shift, dims=2)
        # --- 补丁 4 结束 ---

        # --- (保持 RuntimeError 修复) ---
        inputs_denoised = inputs_denoised.unsqueeze(1)
        # --- 修正结束 ---

        outputs = model_classifier(inputs_denoised)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        total_num += labels.size(0)
        y_true_all.extend(labels.cpu().numpy())
        y_pred_all.extend(torch.max(outputs.data, 1)[1].cpu().numpy())

    train_loss = train_loss / total_num
    train_metrics = cal_metrics_torch(y_true_all, y_pred_all)
    return train_loss, train_metrics


@torch.no_grad()
def test_one_epoch(model_classifier, model_denoiser, test_loader, criterion, device):
    model_classifier.eval()
    test_loss = 0.
    total_num = 0
    y_true_all = []
    y_pred_all = []
    y_scores_all = []

    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # --- 实时降噪 ---
        out = model_denoiser(inputs)
        inputs_denoised = out[0] if isinstance(out, tuple) else out
        # --- 结束 ---

        # --- (保持 RuntimeError 修复) ---
        inputs_denoised = inputs_denoised.unsqueeze(1)
        # --- 修正结束 ---

        outputs = model_classifier(inputs_denoised)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * labels.size(0)
        total_num += labels.size(0)
        y_true_all.extend(labels.cpu().numpy())
        y_pred_all.extend(torch.max(outputs.data, 1)[1].cpu().numpy())
        y_scores_all.extend(F.softmax(outputs, dim=1).cpu().numpy())

    test_loss = test_loss / total_num

    # (保持 AttributeError 修复)
    try:
        y_scores_all_np = np.stack(y_scores_all, axis=0)
    except (ValueError, TypeError):
        y_scores_all_np = np.array([])

    test_metrics = cal_metrics_torch(y_true_all, y_pred_all, y_scores_all_np)
    return test_loss, test_metrics, y_true_all, y_pred_all


def train_test_subject(subject_id, data_dir, model_denoise, device, save_dir_models_temp,
                       model_type, epochs, batch_size, lr, patience, exp_tim):
    # 1. 加载数据
    dataset = LdNetClassificationDataset(data_dir=data_dir, subject_id=subject_id)
    if len(dataset) == 0:
        print(f"受试者 {subject_id} 数据为空，跳过。")
        return None

    # 2. 划分
    train_indices, test_indices = train_test_split(
        list(range(len(dataset))), test_size=0.2, random_state=exp_tim, stratify=dataset.labels_encoded
    )
    train_set = torch.utils.data.Subset(dataset, train_indices)
    test_set = torch.utils.data.Subset(dataset, test_indices)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=0)

    # (保持 SC_BiGRU 修复)
    if model_type == 'SC_BiGRU':
        model_classifier = CNNRNNs(rnn_type='BiGRU').to(device)
    else:
        raise ValueError(f"未知的模型类型: {model_type}")

    # --- 💡 补丁 1：计算类别权重 (解决类别不平衡) ---
    print(f"  > 正在计算 {subject_id} 的类别权重...")
    labels_in_train = []
    for idx in train_set.indices:
        labels_in_train.append(dataset.labels_encoded[idx])

    # (num_classes 来自 common_params.py)
    class_counts = np.bincount(labels_in_train, minlength=num_classes)
    class_counts = np.where(class_counts == 0, 1, class_counts)
    weights = len(labels_in_train) / (num_classes * class_counts)
    weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)
    print(f"  > 类别权重: {np.round(weights, 2)}")
    # --- 补丁 1 结束 ---

    # 3. 工具
    # --- 💡 补丁 1 (应用) ---
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    # --- 💡 补丁 3：添加 Weight Decay (轻正则) ---
    print(f"  > 使用 AdamW 优化器, weight_decay=1e-4")
    optimizer = optim.AdamW(model_classifier.parameters(), lr=lr, weight_decay=1e-4)
    # --- 补丁 3 结束 ---

    # --- 💡 补丁 2：更敏捷的 LR 调度 (Patience=3, Factor=0.5) ---
    scheduler_params = {
        'ReduceLROnPlateau': {
            'mode': 'min',
            'factor': 0.5,  # 每次衰减 50%
            'patience': 3,  # 3 次不降则衰减
            'verbose': False,
            'threshold': 0.0001,
            'min_lr': 1e-6,  # 最小 LR
        }
    }
    scheduler = LrScheduler(optimizer=optimizer, scheduler_type='ReduceLROnPlateau',
                            params=scheduler_params,
                            max_epoch=epochs).get_scheduler()
    # --- 补丁 2 结束 ---

    # 4. 早停
    save_dir_subj = os.path.join(save_dir_models_temp, f'{subject_id}')
    make_dir(save_dir_subj)
    best_model_path = os.path.join(save_dir_subj, f'best_model_{model_type}_exp{exp_tim}.pth')
    early_stopping = EarlyStopping(patience=patience, verbose=True, path=best_model_path)

    # 5. 训练循环
    for epoch in range(1, epochs + 1):
        train_loss, train_metrics = train_one_epoch(
            model_classifier, model_denoise, train_loader, criterion, optimizer, device
        )
        val_loss, val_metrics, _, _ = test_one_epoch(
            model_classifier, model_denoise, test_loader, criterion, device
        )

        scheduler.step(val_loss)
        early_stopping(val_loss, model_classifier)

        if (epoch % 10 == 0) or early_stopping.early_stop:
            print(f"{subject_id} | Epoch {epoch:03d} | "
                  f"Train Loss: {train_loss:.4f}, Acc: {train_metrics['Accuracy']:.2f}% | "
                  f"Val Loss: {val_loss:.4f}, Val Acc: {val_metrics['Accuracy']:.2f}%")

        if early_stopping.early_stop:
            print("早停触发。")
            break

    # 6. 加载最佳模型
    print(f"加载 {subject_id} 的最佳分类模型...")
    best_classifier = torch.load(best_model_path, weights_only=False).to(device)

    test_loss_final, test_metrics_final, y_true_final, y_pred_final = test_one_epoch( # <-- 1. 接收 y_true 和 y_pred
    best_classifier, model_denoise, test_loader, criterion, device
)
# --- 2. 插入混淆矩阵保存逻辑 ---
    print('正在计算并保存混淆矩阵...')
    # (确保 motions_list 在此函数中可用，或者从外部传入)
    # 暂时在此处硬编码 (或者你可以修改函数定义从 单元格 6 传入)
    motions_list_cm = ['WAK', 'STDUP', 'SITDN', 'UPS', 'DNS']

    # (我们复用之前为模型创建的 S{subject_id} 文件夹)
    cm_save_jpg_name = os.path.join(save_dir_subj, f'confusion_matrix_exp{exp_tim}.jpg')
    cm_save_csv_name = os.path.join(save_dir_subj, f'confusion_matrix_exp{exp_tim}.xlsx')

    plot_confusion_matrix = PlotConfusionMatrix(
        y_true_final,
        y_pred_final,
        label_type=motions_list_cm, # <-- 使用动作列表
        show_type='all',
        plot=True,
        save_fig=True,
        save_results=True,
        cmap='YlGnBu'
    )
    plot_confusion_matrix.get_confusion_matrix(cm_save_jpg_name, cm_save_csv_name)
    print(f'混淆矩阵已保存到: {save_dir_subj}')
# --- 混淆矩阵逻辑结束 ---

# 7. 返回最终指标
    test_metrics_final['Subject'] = subject_id
    return test_metrics_final


print("✅ 训练/测试函数 (train_test_subject) 定义完毕。")

In [ ]:
#单元格6
import pandas as pd
import numpy as np

# (来自 common_params.py)
subjects_list = ['Sub01', 'Sub02', 'Sub03', 'Sub04', 'Sub05', 'Sub31', 'Sub32', 'Sub33', 'Sub34', 'Sub35']

# 1. --- 初始化列表 ---
metrics_all_subjects = []

# (确保 单元格 2 已运行)
printlog(f"--- 开始在 {DATASET_NAME} 上执行 {DENOISE_METHOD} + {MODEL_TYPE} 受试者内测试 ---", time=True,
         line_break=True)

# 2. --- 主训练循环 (与你之前的一样) ---
for subject in subjects_list:
    for exp_tim in range(1, K_of_repeated_experiments + 1):

        printlog(f"--- 正在处理受试者: {subject} | 实验: {exp_tim}/{K_of_repeated_experiments} ---", time=True, line_break=True)

        test_metrics = train_test_subject(
            subject_id=subject,
            data_dir=DATA_DIR,
            model_denoise=model_denoise,
            device=DEVICE,
            save_dir_models_temp=save_dir_models_temp,
            model_type=MODEL_TYPE,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            patience=PATIENCE,
            exp_tim=exp_tim
        )

        if test_metrics:
            test_metrics['Experiment'] = exp_tim
            metrics_all_subjects.append(test_metrics)
            print(f"--- 受试者 {subject} 实验 {exp_tim} 完成。最终测试 Acc: {test_metrics['Accuracy']:.2f}% ---")
# --- 主循环结束 ---


printlog(f"--- 所有受试者处理完毕 ---", time=True, line_break=True)

# 3. --- 💡 关键修改：计算并保存为【图片样式】 ---
if metrics_all_subjects:
    # 3.1. 保存【所有50次运行】的详细原始数据
    df_summary = pd.DataFrame(metrics_all_subjects)
    save_path_summary_by_subject = os.path.join(save_dir_root, 'summary_by_subject.csv')
    df_summary.to_csv(save_path_summary_by_subject, index=False, float_format='%.3f')
    print(f"✅ 包含50次运行的详细CSV已保存到: {save_path_summary_by_subject}")

    # 3.2. 计算【每个受试者的平均值】(数字)
    df_avg_by_subject = df_summary.groupby('Subject').mean(numeric_only=True)
    # 计算【每个受试者的标准差】(数字)
    df_std_by_subject = df_summary.groupby('Subject').std(numeric_only=True)

    # 3.3. 创建【受试者10行】的格式化 (mean+std) DataFrame
    df_formatted_subjects = pd.DataFrame(index=df_avg_by_subject.index)

    # 确定要格式化的列 (排除 'Experiment')
    metrics_cols = [col for col in df_avg_by_subject.columns if col != 'Experiment']

    for col in metrics_cols:
        # 格式化为 "均值 + 标准差"，并保留4位小数 (根据图片)
        df_formatted_subjects[col] = df_avg_by_subject[col].map('{:.4f}'.format) + \
                                     "+" + \
                                     df_std_by_subject[col].map('{:.4f}'.format)

    # 3.4. 计算【所有受试者平均值的平均值】(数字)
    df_total_avg = df_avg_by_subject[metrics_cols].mean().to_frame().T

    # 3.5. 计算【所有受试者平均值的标准差】(数字)
    df_total_std = df_avg_by_subject[metrics_cols].std().to_frame().T

    # 3.6. 将总平均值和总标准差格式化为DataFrame (用于拼接)
    # 只保留小数 (根据图片)
    df_total_avg_formatted = df_total_avg.applymap(lambda x: f"{x:.4f}")
    df_total_std_formatted = df_total_std.applymap(lambda x: f"{x:.4f}")

    # 更改索引名以便拼接
    df_total_avg_formatted.index = ['mean']
    df_total_std_formatted.index = ['std']

    # 3.7. 拼接所有部分：10行 (mean+std) + 1行 (total mean) + 1行 (total std)
    df_final_report = pd.concat([df_formatted_subjects, df_total_avg_formatted, df_total_std_formatted])

    # 3.8. 保存最终的报告
    save_path_final_report = os.path.join(save_dir_root, 'all_metrics_averaged_results.csv')
    # index=True 是因为 Subject/mean/std 现在是索引
    df_final_report.to_csv(save_path_final_report, index=True)
    print(f"✅ 【最终报告 (mean+std)】(12行) 已保存到: {save_path_final_report}")

    # (现在 "alone_subject_averaged_results.csv" 已经合并到上面了，不再单独保存)

    # 3.9. 检查 Sub02 (使用未格式化的 df_avg_by_subject)
    try:
        sub02_new_avg = df_avg_by_subject.loc['Sub02']['Accuracy']
        print(f"\n--- 检查 Sub02 ---")
        print(f"Sub02 新的5次实验平均准确率: {sub02_new_avg:.3f}%")
    except Exception as e:
        print(f"无法自动检查 Sub02 的新平均值: {e}")

else:
    print("没有受试者被成功处理。")